In [ ]:
# import statements

import PyPhenom as ppi
import numpy as np
import matplotlib.pyplot as plt
import cv2

from math import factorial
from scipy.fft import fft2, fftshift
from scipy.signal import fftconvolve
from scipy.spatial.distance import pdist, squareform
from lmfit import Model, models, minimize, Minimizer, create_params
from skopt import gp_minimize

%matplotlib widget


In [ ]:
# connect to SEM 

PhenomID = 'MVE086101-20057-F'
username = 'MVE08610120057F'
password = 'KN68X6HJQMT9'

phenom = ppi.Phenom(PhenomID, username, password)

In [ ]:
# global variables

def take_image():
    xdim = 256 # x and y dimensions of image taken
    ydim = 256
    stack = 1
    HDR = False
    scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

    phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim, ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array

    return img_array


def calibrate_kernel_size(type, val_min):
    """
    yea gotta change this

    Parameter wd_val_min: the minimum working distance (float) that the SEM can image at
    """

    sums = -1
    kernel_size = 5
    ori_mag = ppi.MagnificationFromFieldWidth(phenom.GetHFW(), 0.52)

    if type == 'wd':
        phenom.SetHFW(ppi.MagnificationToFieldWidth(10000, 0.52))
        phenom.SetSemWD(val_min)

    elif type == "xst":
        phenom.SetHFW(ppi.MagnificationToFieldWidth(40000, 0.52))
        phenom.SetSemStigmate(ppi.Position(val_min, 0))

    elif type == "yst":
        phenom.SetHFW(ppi.MagnificationToFieldWidth(40000, 0.52))
        phenom.SetSemStigmate(ppi.Position(0, val_min))

    img_array = take_image()

    while np.round(sums, 4) != 0:
        blurred_img = cv2.GaussianBlur(img_array, (kernel_size, kernel_size), 0)
        filtered_data = cv2.Canny(blurred_img, 50, 150)

        sums = filtered_data.sum()
        
        if np.round(sums, 4) != 0:
            kernel_size = kernel_size + 2

    phenom.SetHFW(ppi.MagnificationToFieldWidth(ori_mag, 0.52))
    phenom.SetSemStigmate(ppi.Position(0, 0))

    return kernel_size


wd_val_min = 0.002 #0.008 #0.002
wd_val_max = 0.012 #0.011 #0.025 #0.12198 is max value
xst_val_min = -0.2 #-1
xst_val_max = 0.2 #1
yst_val_min = -0.2 #-1
yst_val_max = 0.2 #1

size = calibrate_kernel_size('wd', wd_val_min)
x_size = calibrate_kernel_size('xst', xst_val_min)
y_size = calibrate_kernel_size('yst', yst_val_min)
print(size, x_size, y_size)



In [ ]:
# autocorrelation functions

def auto_correlation(frame):
    """
    Returns the computed auto_correlation for a given frame.

    Parameter frame: an SEM image
    """

    ft = np.fft.fft2(frame)
    autocorr = np.fft.ifft2(ft * np.conjugate(ft))
    return np.fft.fftshift(autocorr).real


def filtered_autocorr(img_array, kernel_size):
    """
    Returns the autocorrelation array for the given image array and the kernel size to blur the image.

    Parameter img_array: a Numpy array
    Parameter kernel_size: an int specifying the size of the blurring kernel
    """
    
    blurred_img = cv2.GaussianBlur(img_array, (kernel_size, kernel_size), 0) # blurs the image to help with noise
    filtered_data = cv2.Canny(blurred_img, 50, 150) # sets the values of edges = 255, everything else = 0
    acdata = auto_correlation(filtered_data) # calculates the auto_correlation data

    return acdata

In [ ]:
# fitting functions

# Define basic Zernike polynomials (on the unit disk)
def zernike_radial(n, m, rho):
    R = np.zeros_like(rho)
    for k in range((n - abs(m)) // 2 + 1):
        coeff = (-1)**k * factorial(n - k)
        coeff /= (factorial(k) *
                  factorial((n + abs(m)) // 2 - k) *
                  factorial((n - abs(m)) // 2 - k))
        R += coeff * rho**(n - 2 * k)
    return R


def zernike(n, m, rho, theta):
    if m > 0:
        return zernike_radial(n, m, rho) * np.cos(m * theta)
    elif m < 0:
        return zernike_radial(n, -m, rho) * np.sin(-m * theta)
    else:
        return zernike_radial(n, 0, rho)


# Mapping Noll index to (n, m)
NOLL_MAP = {
    1:  (0,  0),   # Piston (constant phase offset)
    2:  (1, -1),   # Tilt X (Tip)
    3:  (1,  1),   # Tilt Y (Tilt)
    4:  (2, -2),   # Oblique Astigmatism
    5:  (2,  0),   # Defocus
    6:  (2,  2),   # Vertical Astigmatism
    7:  (3, -3),   # Vertical Trefoil
    8:  (3, -1),   # Vertical Coma
    9:  (3,  1),   # Horizontal Coma
    10: (3,  3),   # Horizontal Trefoil
    11: (4, -4),   # Oblique Tetrafoil
    12: (4, -2),   # Oblique Secondary Astigmatism
    13: (4,  0),   # Primary Spherical Aberration
    14: (4,  2),   # Vertical Secondary Astigmatism
    15: (4,  4),   # Vertical Tetrafoil
}


def incoherent_zernike_psf(N=256, defocus = None, stig = None): # zernike_coeffs=None):
    if defocus is None: # if zernike_coeffs is None:
        zernike_coeffs = {
            5:2,  # defocus
        }

    else:
        zernike_coeffs = {
            5:defocus
        }

    if stig is not None:
        zernike_coeffs[6] = stig

    x = np.linspace(- 1, 1, N)
    X, Y = np.meshgrid(x, x)
    rho = np.sqrt(X**2 + Y**2)
    theta = np.arctan2(Y, X)

    aperture = rho <= 1 #boolean array

    wavefront = np.zeros_like(rho)

    for j, coeff in zernike_coeffs.items():
        n, m = NOLL_MAP[j]
        Z = zernike(n, m, rho, theta)
        Z[~aperture] = 0
        wavefront += coeff * Z

    # Pupil function with aberration
    phase = 2 * np.pi * wavefront # this could be scaled by the wavelength.
    # suggest to not to do this in this case that adds to the confusion ==> this is not the same as a coherent point source!

    pupil = np.exp(1j * phase) 
    pupil[~aperture] = 0

    # Coherent PSF
    h = fftshift(fft2(pupil)) #The point spread function (PSF), which is given by the square of the absolute value of the 
    psf_coh = np.abs(h)**2    #Fourier transform of the wavefront, shows what the intensity image of a point source would look like.

    # Incoherent PSF: autocorrelation of coherent intensity
    psf_incoh = fftconvolve(psf_coh, psf_coh, mode='same')
    psf_incoh /= psf_incoh.sum()

    return psf_incoh


def incoh_psf_fit(ac, df, st):
    model = Model(incoherent_zernike_psf)

    nrows = ac.shape[0]
    ncols = ac.shape[1]

    pars = create_params(N = dict(value = 256, vary = False),
                         defocus = dict(value = df, vary = True, min = 0),
                         stig = dict(value = st, vary = True))

    result = model.fit(ac, params = pars) # uses the above parameters and  model to fit the data

    # plt.figure()
    # plt.imshow(ac)
    # plt.contour(result.best_fit)

    return result

In [ ]:
# optimization functions

def optimum_wd(wd, kernel_size = size):
    """
    Returns the negative sum of the autocorrelation values for the four pixels surrounding the center pixel given pars, 
    a Parameters dictionary containing the working distance range to calculate over.
    The objective function to minimize for working distance.

    Parameter phenom: a Phenom object
    Parameter wd: a float specifying the working distance 
    Parameter kernel_size: an int specifying the size of the blurring kernel
    """

    print(wd)
    phenom.SetSemWD(wd[0])

    img_array = take_image()
    acdata = filtered_autocorr(img_array, kernel_size)

    sums = (acdata[acdata.shape[0] // 2 - 1, acdata.shape[1] // 2] + 
            acdata[acdata.shape[0] // 2 + 1, acdata.shape[1] // 2] + 
            acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] + 
            acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 + 1])
    
    return sums * (-1)


def auto_focus(wd_val_min, wd_val_max):
    """
    Sets the working distance to the value corresponding to a focused image.

    Parameter wd_val_min: the minimum working distance (float) that is considered when finding the optimal working distance
    Parameter wd_val_max: the maximum working distance (float) that is considered when finding the optimal working distance
    """

    res = gp_minimize(optimum_wd, 
                      [(wd_val_min, wd_val_max)], 
                      acq_func = "EI", 
                      n_calls = 10, 
                      n_initial_points = 5, 
                      verbose = False, 
                      random_state = 1234) 

    phenom.SetSemWD(res.x[0])
    return res.x[0]


def optimum_xstig(xst, kernel_size = x_size):
    print(xst)
    phenom.SetSemStigmate(ppi.Position(xst[0], 0))

    img_array = take_image()
    # acdata = filtered_autocorr(img_array, kernel_size)

    # sums = (acdata[acdata.shape[0] // 2 - 1, acdata.shape[1] // 2] + 
    #         acdata[acdata.shape[0] // 2 + 1, acdata.shape[1] // 2] + 
    #         acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] + 
    #         acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 + 1])
    
    # return sums * (-1)

    acdata = auto_correlation(img_array)

    plt.figure()
    plt.subplot(1, 3, 1)
    plt.imshow(acdata)

    if acdata.sum() != 0:
        threshold = ((acdata.max() - acdata.min()) * 0.25) + acdata.min()
        acdata[acdata < threshold] = threshold
        acdata = acdata - threshold
        if acdata.sum() != 0:
            acdata /= acdata.sum()

    plt.subplot(1, 3, 2)
    plt.imshow(acdata)

    result = incoh_psf_fit(acdata, 1, 2)

    plt.subplot(1, 3, 3)
    plt.imshow(result.best_fit)

    return result.best_values['stig']


def optimum_ystig(yst, kernel_size = y_size):
    phenom.SetSemStigmate(ppi.Position(0, yst[0]))

    img_array = take_image()
    # acdata = filtered_autocorr(img_array, kernel_size)

    # sums = (acdata[acdata.shape[0] // 2 - 1, acdata.shape[1] // 2] + 
    #         acdata[acdata.shape[0] // 2 + 1, acdata.shape[1] // 2] + 
    #         acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] + 
    #         acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 + 1])
    
    # return sums * (-1)

    acdata = auto_correlation(img_array)

    plt.figure()
    plt.subplot(1, 3, 1)
    plt.imshow(acdata)

    if acdata.sum() != 0:
        threshold = ((acdata.max() - acdata.min()) * 0.25) + acdata.min()
        acdata[acdata < threshold] = threshold
        acdata = acdata - threshold
        if acdata.sum() != 0:
            acdata /= acdata.sum()

    plt.subplot(1, 3, 2)
    plt.imshow(acdata)

    result = incoh_psf_fit(acdata, 1, 2)

    plt.subplot(1, 3, 3)
    plt.imshow(result.best_fit)

    return result.best_values['stig']


def auto_stig(xst_val_min, xst_val_max, yst_val_min, yst_val_max):
    x_res = gp_minimize(optimum_xstig, 
                        [(xst_val_min, xst_val_max)], 
                        acq_func = "EI", 
                        n_calls = 15, 
                        n_initial_points = 5, 
                        verbose = False, 
                        random_state = 1234) 

    y_res = gp_minimize(optimum_ystig, 
                        [(yst_val_min, yst_val_max)], 
                        acq_func = "EI", 
                        n_calls = 15, 
                        n_initial_points = 5, 
                        verbose = False, 
                        random_state = 1234) 

    phenom.SetSemStigmate(ppi.Position(x_res.x[0], y_res.x[0]))

    return x_res.x[0], y_res.x[0]

In [ ]:
# TEST THIS FIRST !!!

def recenter():
    # run autofocus and thatll set the wd to the correct one at small mags

    # locating the area with features in it to focus on
    img_array = take_image()
    blurred_img = cv2.GaussianBlur(img_array, (size, size), 0) 
    filtered = cv2.Canny(blurred_img, 50, 150)

    points = [(i, j) for i, row in enumerate(filtered) for j, val in enumerate(row) if val == 255]
    distances = squareform(pdist(points, metric='euclidean'))
    total_distances = [sum(row) for row in distances]

    # total_distances = []
    # for row in range(len(distances)):
    #     total_distances.append(distances[row].sum())

    new_center = points[np.argmin(total_distances)]

    print(new_center)

    plt.figure()
    plt.imshow(filtered)

    specific_keypoint = cv2.KeyPoint(x=new_center[0], y=new_center[1], size=10, angle=0, response=0, octave=0, class_id=0)
    keypoints_to_draw = [specific_keypoint]
    img3 = cv2.drawKeypoints(img_array, keypoints_to_draw, None, color=(0,255,0), flags=0)
    plt.figure()
    plt.imshow(img3), plt.show()

    # move to new_center
    center = (img_array.shape[0] // 2, img_array.shape[1] // 2)
    px_distance = [center[0] - new_center[0], center[1] - new_center[1]]
    scale = 3840 / phenom.GetHFW() 
    m_distance = [val / scale for val in px_distance]

    phenom.MoveBy(m_distance[0], m_distance[1], algorithm = NavigationAlgorithm.Auto)


    # with orb
    # img = take_image()
    # orb = cv2.ORB_create()
    # kp = orb.detect(img,None)
    # kp, des = orb.compute(img, kp)

    # img2 = cv2.drawKeypoints(img, kp, None, color=(0,255,0), flags=0)
    # plt.figure()
    # plt.imshow(img2), plt.show()

    # points = []
    # for keypoint in kp:
    #     points.append(keypoint.pt)

    # pairwise_distances = squareform(pdist(points, metric='euclidean'))
    # sums = []
    # for row in range(len(pairwise_distances)):
    #     sums.append(pairwise_distances[row].sum())

    # specific_keypoint = cv2.KeyPoint(x=kp[np.argmin(sums)].pt[0], y=kp[np.argmin(sums)].pt[1], size=10, angle=0, response=0, octave=0, class_id=0)
    # keypoints_to_draw = [specific_keypoint]

    # img3 = cv2.drawKeypoints(img, keypoints_to_draw, None, color=(0,255,0), flags=0)
    # plt.figure()
    # plt.imshow(img3), plt.show()

    

In [ ]:
def thingy(wd_val_min, wd_val_max, xst_val_min, xst_val_max, yst_val_min, yst_val_max):

    # for magnification in [10000, 20000, 40000, 80000]:
    #     phenom.SetHFW(ppi.MagnificationToFieldWidth(magnification, 0.52))
    #     curr = auto_focus(wd_val_min, wd_val_max)

    #     print("curr", curr)

    phenom.SetHFW(ppi.MagnificationToFieldWidth(10000, 0.52))
    curr = auto_focus(wd_val_min, wd_val_max)

    print("curr", curr)

    phenom.SetHFW(ppi.MagnificationToFieldWidth(120000, 0.52))
    x, y = auto_stig(xst_val_min, xst_val_max, yst_val_min, yst_val_max)

    print('stig', x, y)

    for r in range(2):
        phenom.SetHFW(ppi.MagnificationToFieldWidth(80000, 0.52))
        auto_focus(curr - (1 / 1000), curr + (1 / 1000))

        phenom.SetHFW(ppi.MagnificationToFieldWidth(120000, 0.52))
        auto_stig(x - (0.1 / 20), x + (0.1 / 20), y - (0.1 / 20), y + (0.1 / 20))


thingy(wd_val_min, wd_val_max, xst_val_min, xst_val_max, yst_val_min, yst_val_max)

In [ ]:
def whole_thing2(wd_val_min, wd_val_max, xst_val_min, xst_val_max, yst_val_min, yst_val_max):
    prev = None
    curr = None
    ov_prev = None
    ov_curr = None
    res = 1

    curr_min_wd = wd_val_min
    curr_max_wd = wd_val_max
    curr_min_xst = xst_val_min
    curr_max_xst = xst_val_max
    curr_min_yst = yst_val_min
    curr_max_yst = yst_val_max

    # while ov_prev is None or np.abs(ov_curr - ov_prev) >= 0.000001:
    for i in range(2):
        magnification = 10000

        ov_prev = ov_curr

        while prev is None or res >= 0.001:
            prev = curr
            phenom.SetHFW(ppi.MagnificationToFieldWidth(magnification, 0.52))

            curr = auto_focus(curr_min_wd, curr_max_wd)

            print("curr", curr)
            
            curr_min_wd = curr - (res / 1000)
            curr_max_wd = curr + (res / 1000)

            res /= 10
            magnification *= 2

        prev = None
        curr = None
        ov_curr = curr

        phenom.SetHFW(ppi.MagnificationToFieldWidth(120000, 0.52))
        x, y = auto_stig(curr_min_xst, curr_max_xst, curr_min_yst, curr_max_yst)

        print('stig', x, y)

        curr_min_xst = x - (0.1 / 20)
        curr_max_xst = x + (0.1 / 20)
        curr_min_yst = y - (0.1 / 20)
        curr_max_yst = y + (0.1 / 20)

        res = 1


whole_thing2(wd_val_min, wd_val_max, xst_val_min, xst_val_max, yst_val_min, yst_val_max)

In [ ]:
curr_min_wd = wd_val_min
curr_max_wd = wd_val_max
curr_min_xst = xst_val_min
curr_max_xst = xst_val_max
curr_min_yst = yst_val_min
curr_max_yst = yst_val_max

x, y = auto_stig(curr_min_xst, curr_max_xst, curr_min_yst, curr_max_yst)